#### LIBRARY IMPORTS

In [121]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Neural Netowrk specifc imports
import copy # for deepcopy in save_results()

import torch
import torch.nn as nn # NN layers and loss functions
import torch.optim as optim # Optimization Algorithms
# Batching Data:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

#### CONFIGURATOINS

In [122]:
FEATURE_ROOT = "eGeMAPs_features"
OUTPUT_ROOT = "outputs-undiarized"
RANDOM_SEED = 7
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### FUNCTIONS

In [123]:
# Early stopping
PATIENCE = 10 # Stop after 10 epochs of no improvement in validation loss

In [124]:
def load_participant_data(participant_folder):
    csv_files = sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv")) # For multiple files

    # print(df.head())
    # print(df.shape)

    # Train / Validation / Test Split
    
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    
    # # 80% train, 20% temp

    # Read every session and combine them into one dataframe
    full_df = pd.concat(
        [pd.read_csv(f) for f in csv_files],
        ignore_index=True
    )   
    print(f"Total samples: {len(full_df)}")
    
    # Remove unnecessary columns
    drop_cols = ["Unnamed: 0", "start_time", "end_time"]
    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )
    print(full_df.columns)

    # Seperate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    # First split: Train vs Temp
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        train_size=TRAIN_RATIO,
        random_state=RANDOM_SEED,
        shuffle=True
        stratify=y
    )

    # Second split: Validation vs Test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=TEST_RATIO/(VAL_RATIO+TEST_RATIO),
        random_state=RANDOM_SEED,
        shuffle=True
        stratify=y
    )

    # Print unique labels for debugging
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Testing samples: {len(X_test)}")

    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))
    
    return X_train, X_val, X_test, y_train, y_val, y_test

In [125]:
def preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_val = encoder.transform(y_val)
    y_test = encoder.transform(y_test)

    # Print for debugging purpose
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_val = scaler.transform(X_val) # Transform the data using the learned mean and std
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        scaler,
        encoder
    )  

In [126]:
class NeuralNetwork(nn.Module):

    # Define the architecture of the neural network ; Constructor
    def __init__(self, input_dim):

        super().__init__() # Initialize the parent class (nn.Module) first, then inherit functionalities

        self.network = nn.Sequential(
            # First Hidden Layer 88 -> 128 nueruons
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Second Hidden Layer 128 -> 64 nueruons
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output Layer 64 -> 1 nuerons
            nn.Linear(64, 1)

        )
    
    # Forward pass through the network; Automatically called  when calling model; Then return the model predictions
    def forward(self, x):
        return self.network(x)


# Build the model, then move it to GPU/ CPU and print the model architecture
def build_model(input_dim):
    # Build model
    model = NeuralNetwork(input_dim)
    
    # Move model to GPU/ CPU
    model.to(DEVICE)
    
    print(model)

    return model

In [127]:
def train_model(
        model,
        train_loader,
        val_loader
):
    # BCEWithLogitsLoss is good for our case of binary classification
    # It performs sigmoid + Binary Cross Entropy Loss 
    criterion = nn.BCEWithLogitsLoss()

    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Initialize best loss to be infinity and patience counter to 0
    best_loss = float("inf")
    patience_counter = 0

    # Store training and validation loss for each epoch (perhaps for plotting later)
    history = {
        "train_loss": [],
        "val_loss": []
    }

    # Epoch Loop; Max is 100, but may stop earlier due to early stopping
    for epoch in range(100):

        #TRAINING

        # Activate training (Dropout layers and gradient computation)
        model.train()

        train_loss = 0

        # Train Batch-wise (32)
        for X_batch, y_batch in train_loader:
            
            # Move batch to GPU/ CPU
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            # Reset the gradients before backpropagation
            optimizer.zero_grad()

            # Foeward pass prediction
            outputs = model(X_batch)

            # Compute loss
            loss = criterion(outputs, y_batch)

            # Backpropagation 
            # Computes gradients
            loss.backward()
            # Update weights using optimizer
            optimizer.step()

            train_loss += loss.item() # loss is a tensor

        # Average traingin loss per batch
        train_loss /= len(train_loader)

        # VALIDATION
        
        # Turn off training (Dropout layers and gradient computation)
        model.eval()

        val_loss = 0

        # Disable gradient computation
        with torch.no_grad():
            # Validate Batch-wise (32)
            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)

                outputs = model(X_batch)

                loss = criterion(outputs, y_batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"Train={train_loss:.4f} "
            f"Val={val_loss:.4f}"
        )

        # If validation loss improves, save the model weights and reset patience counter
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

        # If validation loss does not improve, increment patience counter and check for early stopping
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping")
                break

    # Restore the best model weights after training is complete
    model.load_state_dict(best_weights)

    return history

In [128]:
def evaluate_model(
        model,
        test_loader,
        encoder
):
    # No dropouts
    model.eval()

    probabilities = []
    predictions = []
    actual = []

    # Disable gradient computation
    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(DEVICE)

            # Forward pass prediction
            outputs = model(X_batch)
            
            # Apply sigmoid to get probabilities
            probs = torch.sigmoid(outputs)

            # Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
            preds = (probs >= 0.5).float()

            # NumPy cannotread GPU tensors, so we need to move them to CPU and convert to NumPy arrays before storing them in lists
            probabilities.extend(probs.cpu().numpy().flatten())
            predictions.extend(preds.cpu().numpy().flatten())
            
            actual.extend(y_batch.numpy().flatten())

    # Convert lists to NumPy arrays and ensure they are of integer type
    predictions = np.array(predictions).astype(int)
    actual = np.array(actual).astype(int)

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions)
    }

    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [129]:
''' The following are saved:
    Model weights (model.pt)
    Scaler (scaler.pkl)
    Encoder (encoder.pkl)
    Training History (history.csv)
    Metrics (metrics.csv)
    Confusion Matrix (confusion_matrix.csv)
    Classification Report (classification_report.csv) '''
    
def save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    ):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    torch.save(model.state_dict(),participant_output / "model.pt")
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save History
    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(participant_output, "history.csv"),index=False)
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [130]:
summary_results = []

participants = sorted(os.listdir(FEATURE_ROOT))

for participant in participants:

    print(f"Training {participant}")
    
    # Load data
    X_train, X_val, X_test, y_train, y_val, y_test = load_participant_data(participant)
    
    # Preprocess data
    X_train, X_val, X_test, y_train, y_val, y_test, scaler, encoder = preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test)
    
    # Convert NumPy arrays into PyTorch tensors.
    X_train_tensor = torch.FloatTensor(X_train)
    X_val_tensor = torch.FloatTensor(X_val)
    X_test_tensor = torch.FloatTensor(X_test)

    # Shape of y_train, y_val, y_test is (N,), but we need (N,1) for BCEWithLogitsLoss
    y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
    y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)
    y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

    # Create TensorDatasets; Join features + labels
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    
    # Create DataLoaders; Batch the data
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True # Shuffle training data for better generalization
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False
    )

    # Build model
    model = build_model(X_train.shape[1])
    
    # Train model and return training history
    history = train_model(model, train_loader, val_loader)
    
    # Evaluate model and return evaluation metrics
    evaluation = evaluate_model(model, test_loader, encoder)
    
    # Save results
    save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    )
    
    # Append results to summary
    summary_results.append({
        "Participant": participant,
        **evaluation["metrics"]
    })

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    

Training p11
p11: 9 CSV files
Total samples: 811
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', 'mfcc1_sma